<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-1-deep-learning/lab-04-the-tuning-study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 4 (graded) — The tuning study
**Course 1: Hands-On Deep Learning with Python — Chapter 4: Optimization & regularization deep-dive**

**Problem brief (Leo Farkas, Orbit Retail, continued):** "Your first model overfits and
training is unstable. Make it reliable enough to trust in a weekly batch job."

**Fixed compute budget for this lab: 30 epochs per run, no exceptions** — the point is to get
more from the same budget through better configuration, not to just train longer.

**What you'll submit:** the MLflow sweep below, a coarse-to-fine narrative (markdown cells,
fill them in as you go), and your final config with justification.

In [ ]:
!pip install -q mlflow

In [ ]:
import io
import urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import mlflow
from sklearn.metrics import roc_auc_score

np.random.seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

def load_churn_data():
    try:
        # fetch with a bounded timeout first - pd.read_csv(url) has no timeout of its own
        # and can hang the whole cell indefinitely on a stalled connection
        req = urllib.request.Request(
            'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/'
            'master/data/Telco-Customer-Churn.csv',
            headers={'User-Agent': 'aibits-course-lab/1.0'},
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            raw = resp.read()
        df = pd.read_csv(io.BytesIO(raw))
    except Exception:
        n = 3000
        tenure = np.random.randint(0, 72, n)
        monthly = np.random.uniform(18, 120, n)
        contract = np.random.choice(['Month-to-month', 'One year', 'Two year'], n, p=[0.55, 0.25, 0.2])
        risk = 1 / (1 + np.exp(-(2.0 - 0.04 * tenure - 0.01 * monthly + (contract == 'Month-to-month') * 1.2)))
        df = pd.DataFrame({'tenure': tenure, 'MonthlyCharges': monthly, 'Contract': contract,
                            'Churn': np.where(np.random.rand(n) < risk, 'Yes', 'No')})
    df['Churn'] = (df['Churn'] == 'Yes').astype(int)
    df['MonthlyCharges'] = pd.to_numeric(df['MonthlyCharges'], errors='coerce')
    return df.dropna(subset=['MonthlyCharges'])

df = pd.get_dummies(load_churn_data(), columns=['Contract'], drop_first=True)
feature_cols = ['tenure', 'MonthlyCharges'] + [c for c in df.columns if c.startswith('Contract_')]
X = df[feature_cols].to_numpy(dtype=np.float32)
X = (X - X.mean(0)) / (X.std(0) + 1e-8)
y = df['Churn'].to_numpy(dtype=np.float32)
n = len(X); idx = np.random.permutation(n); n_train = int(n * 0.75)
X_train, y_train = X[idx[:n_train]], y[idx[:n_train]]
X_val, y_val = X[idx[n_train:]], y[idx[n_train:]]

class ChurnDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

## 1. The deliberately broken baseline
Huge learning rate, no normalization, no regularization, a needlessly deep/wide network for
this little data. Run it and watch it misbehave — this is your starting point.

In [ ]:
class BrokenMLP(nn.Module):
    def __init__(self, n_in):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 256), nn.ReLU(),
            nn.Linear(256, 1),
        )
    def forward(self, x): return self.net(x)


def run_config(model_fn, lr, weight_decay=0.0, batch_size=64, n_epochs=30,
               optimizer_name='SGD', scheduler=None, grad_clip=None, seed=0):
    torch.manual_seed(seed)
    model = model_fn(X_train.shape[1]).to(device)
    opt_cls = {'SGD': torch.optim.SGD, 'Adam': torch.optim.Adam, 'AdamW': torch.optim.AdamW}[optimizer_name]
    optimizer = opt_cls(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = None
    if scheduler == 'cosine':
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    loss_fn = nn.BCEWithLogitsLoss()
    loader = DataLoader(ChurnDataset(X_train, y_train), batch_size=batch_size, shuffle=True)

    train_losses, val_losses = [], []
    X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
    y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1).to(device)

    for epoch in range(n_epochs):
        model.train()
        epoch_loss, nb = 0.0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            if grad_clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            epoch_loss += loss.item(); nb += 1
        if sched: sched.step()
        train_losses.append(epoch_loss / nb)
        model.eval()
        with torch.no_grad():
            val_losses.append(loss_fn(model(X_val_t), y_val_t).item())

    model.eval()
    with torch.no_grad():
        val_prob = torch.sigmoid(model(X_val_t)).cpu().numpy().ravel()
    auc = roc_auc_score(y_val, val_prob)
    return {'train_losses': train_losses, 'val_losses': val_losses, 'final_val_loss': val_losses[-1], 'val_auc': auc}

baseline = run_config(BrokenMLP, lr=1.0, optimizer_name='SGD')
print(f"Baseline — final val loss: {baseline['final_val_loss']:.4f}  AUC: {baseline['val_auc']:.4f}")
print('If val loss is huge, NaN, or oscillating, that\u2019s the broken baseline working as intended.')

### Round 1 narrative (fill in)
What's wrong with the baseline, and what's the single most important thing to fix first?

_Your answer here._

## 2. A regularized, right-sized model
Add BatchNorm, Dropout, a smaller network, and switch to AdamW. Sweep the learning rate
coarsely (log-scale) first, then narrow.

In [ ]:
class RegularizedMLP(nn.Module):
    def __init__(self, n_in, hidden=32, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
    def forward(self, x): return self.net(x)

mlflow.set_experiment('orbit-retail-tuning-study')

# TODO: coarse LR sweep, log-scale — try e.g. [1e-1, 1e-2, 1e-3, 1e-4]
coarse_lrs = [1e-1, 1e-2, 1e-3, 1e-4]
coarse_results = []
for lr in coarse_lrs:
    with mlflow.start_run(run_name=f'coarse-lr-{lr}'):
        mlflow.log_params({'stage': 'coarse', 'lr': lr, 'optimizer': 'AdamW'})
        r = run_config(RegularizedMLP, lr=lr, weight_decay=1e-4, optimizer_name='AdamW')
        mlflow.log_metrics({'final_val_loss': r['final_val_loss'], 'val_auc': r['val_auc']})
        coarse_results.append({'lr': lr, **{k: v for k, v in r.items() if k not in ('train_losses', 'val_losses')}})
        print(f"lr={lr:<8} val_loss={r['final_val_loss']:.4f}  AUC={r['val_auc']:.4f}")

pd.DataFrame(coarse_results)

### Round 2 narrative (fill in)
Which order of magnitude looked best? Now narrow the search around it in the next cell.

In [ ]:
# TODO: fine LR sweep around your best coarse value, e.g. [best/3, best, best*3]
fine_lrs = [3e-3, 1e-3, 3e-4]  # <- adjust based on your coarse sweep result
fine_results = []
for lr in fine_lrs:
    with mlflow.start_run(run_name=f'fine-lr-{lr}'):
        mlflow.log_params({'stage': 'fine', 'lr': lr, 'optimizer': 'AdamW', 'scheduler': 'cosine'})
        r = run_config(RegularizedMLP, lr=lr, weight_decay=1e-4, optimizer_name='AdamW', scheduler='cosine')
        mlflow.log_metrics({'final_val_loss': r['final_val_loss'], 'val_auc': r['val_auc']})
        fine_results.append({'lr': lr, **{k: v for k, v in r.items() if k not in ('train_losses', 'val_losses')}})
        print(f"lr={lr:<8} val_loss={r['final_val_loss']:.4f}  AUC={r['val_auc']:.4f}")

pd.DataFrame(fine_results)

In [ ]:
import matplotlib.pyplot as plt

best_lr = fine_lrs[int(np.argmin([r['final_val_loss'] for r in fine_results]))]
final = run_config(RegularizedMLP, lr=best_lr, weight_decay=1e-4, optimizer_name='AdamW', scheduler='cosine')
plt.plot(final['train_losses'], label='train')
plt.plot(final['val_losses'], label='val')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(); plt.title(f'Final config (lr={best_lr})')
plt.show()
print(f"Final val loss: {final['final_val_loss']:.4f}  AUC: {final['val_auc']:.4f}")
print(f"Improvement vs. broken baseline: {baseline['final_val_loss']:.4f} -> {final['final_val_loss']:.4f}")

## 3. Final config and justification (fill in)
State your final config (lr, optimizer, regularization, schedule) and justify each choice
from what the sweeps above actually showed — not from general folklore.

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 4: Optimization & regularization deep-dive*